Data preprocessing

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [2]:
spark = (
    SparkSession.builder
    .appName("BusServiceReliability")
    .master("local[*]")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/01 22:15:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/01 22:15:54 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/08/01 22:15:54 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [4]:
df = spark.read.parquet("../outputs/timetable_parquet")

In [5]:
df.show(5, truncate=False)

+-------------------+--------------------------------------------------------------------------------------------------------+---------+-------------------+------------------+---------------+----------+--------------------------------------------------+-------------+
|creation_time      |filename                                                                                                |line_name|modified_time      |number_of_journeys|number_of_stops|operator  |route_description                                 |service_code |
+-------------------+--------------------------------------------------------------------------------------------------------+---------+-------------------+------------------+---------------+----------+--------------------------------------------------+-------------+
|2022-08-19T12:05:39|X2-None--SCMY-CZ-2026-03-29-Chester_July_2026_(SchOut)__SCMY_PC1033334_224_20260719-BODS_V1_1.xml       |2        |2026-07-02T13:55:43|141               |215            |Stage

## Dataset Overview

In [6]:
df.printSchema()


root
 |-- creation_time: string (nullable = true)
 |-- filename: string (nullable = true)
 |-- line_name: string (nullable = true)
 |-- modified_time: string (nullable = true)
 |-- number_of_journeys: long (nullable = true)
 |-- number_of_stops: long (nullable = true)
 |-- operator: string (nullable = true)
 |-- route_description: string (nullable = true)
 |-- service_code: string (nullable = true)



## Dataset Dimensions

In [7]:
print("Number of rows:", df.count())
print("Number of columns:", len(df.columns))

Number of rows: 123
Number of columns: 9


## Missing Value Analysis

In [8]:
from pyspark.sql.functions import col, count, when

missing_values = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

missing_values.show()

+-------------+--------+---------+-------------+------------------+---------------+--------+-----------------+------------+
|creation_time|filename|line_name|modified_time|number_of_journeys|number_of_stops|operator|route_description|service_code|
+-------------+--------+---------+-------------+------------------+---------------+--------+-----------------+------------+
|            0|       0|        0|            0|                 0|              0|       0|                0|           0|
+-------------+--------+---------+-------------+------------------+---------------+--------+-----------------+------------+



## Duplicate Record Analysis

In [9]:
total_rows = df.count()
unique_rows = df.dropDuplicates().count()

print("Total rows:", total_rows)
print("Unique rows:", unique_rows)
print("Duplicate rows:", total_rows - unique_rows)

Total rows: 123
Unique rows: 123
Duplicate rows: 0


## Data Type Conversion

In [11]:
from pyspark.sql.functions import to_timestamp

df = (
    df.withColumn("creation_time", to_timestamp("creation_time"))
      .withColumn("modified_time", to_timestamp("modified_time"))
)

In [12]:
df.printSchema()

root
 |-- creation_time: timestamp (nullable = true)
 |-- filename: string (nullable = true)
 |-- line_name: string (nullable = true)
 |-- modified_time: timestamp (nullable = true)
 |-- number_of_journeys: long (nullable = true)
 |-- number_of_stops: long (nullable = true)
 |-- operator: string (nullable = true)
 |-- route_description: string (nullable = true)
 |-- service_code: string (nullable = true)



## Feature Engineering

In [13]:
from pyspark.sql.functions import when

df = df.withColumn(
    "route_size",
    when(df.number_of_stops < 50, "Short")
    .when(df.number_of_stops < 100, "Medium")
    .otherwise("Long")
)

In [14]:
df.select(
    "line_name",
    "number_of_stops",
    "route_size"
).show(10, truncate=False)

+---------+---------------+----------+
|line_name|number_of_stops|route_size|
+---------+---------------+----------+
|2        |215            |Long      |
|51       |32             |Short     |
|X30      |81             |Medium    |
|772      |37             |Short     |
|53       |83             |Medium    |
|726      |44             |Short     |
|617      |22             |Short     |
|881      |52             |Medium    |
|58       |38             |Short     |
|716      |43             |Short     |
+---------+---------------+----------+
only showing top 10 rows


## Save Cleaned Dataset

In [16]:
df.write.mode("overwrite").parquet("../outputs/cleaned_timetable_parquet")

print("Cleaned dataset saved successfully!")

26/08/01 22:22:17 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/08/01 22:22:17 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/08/01 22:22:17 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/08/01 22:22:17 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/08/01 22:22:17 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers


Cleaned dataset saved successfully!


26/08/01 22:22:17 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/08/01 22:22:17 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/08/01 22:22:17 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/08/01 22:22:17 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [17]:
cleaned_df = spark.read.parquet("../outputs/cleaned_timetable_parquet")

cleaned_df.show(5, truncate=False)

+-------------------+--------------------------------------------------------------------------------------------------------+---------+-------------------+------------------+---------------+----------+--------------------------------------------------+-------------+----------+
|creation_time      |filename                                                                                                |line_name|modified_time      |number_of_journeys|number_of_stops|operator  |route_description                                 |service_code |route_size|
+-------------------+--------------------------------------------------------------------------------------------------------+---------+-------------------+------------------+---------------+----------+--------------------------------------------------+-------------+----------+
|2022-08-19 12:05:39|X2-None--SCMY-CZ-2026-03-29-Chester_July_2026_(SchOut)__SCMY_PC1033334_224_20260719-BODS_V1_1.xml       |2        |2026-07-02 13:55:43|141    

# Summary

In this notebook, the timetable dataset was successfully preprocessed. Missing values and duplicate records were checked, data types were converted to appropriate formats, and a new feature (`route_size`) was created based on the number of stops. The cleaned dataset was then saved in Parquet format for use in the following notebooks.